# 第 02 章 · 上下文：决定每一步让模型看到什么

修复 task-list 时，模型需要看到任务约束、失败证据和最近的尝试；测试输出却可能占掉整个窗口。
因此“历史里保存什么”与“下一次请求带什么”必须分开。历史保存发生过的事实，投影负责选择，
摘要只是可替换的派生结果。删掉派生投影，仍应能从历史重新构造请求。

本章沿 **原始历史 → 有预算的投影 → 摘要候选 → 版本检查后安装** 前进。
你会改变预算并制造一次摘要晚到，观察为什么压缩不是简单的列表截断。
字节预算用于确定性教学，不能替代 tokenizer，也不包含请求协议开销。

**运行边界**：本章使用最终包的共享组件，逐格验证本专题机制；完整任务运行见第 31 章。所有代码使用 Python 标准库；不调用真实模型、不读取凭据、不自动安装软件。按顺序运行；需要磁盘的实验只在 `fresh_run_dir()` 新建目录内写入。修改参数后，以重新执行产生的断言与输出为准。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter


In [2]:
ctx = load_example('02', 'context')

def expect_error(error_type, operation, message):
    try:
        operation()
    except error_type as error:
        assert message in str(error), str(error)
        return str(error)
    raise AssertionError(f'应出现 {error_type.__name__}: {message}')


## 1. 历史快照不能成为第二个写入口


    先登记“不能改验证器”作为受保护原文，再加入一对工具调用与输出。调用和输出通过 `call_id` 关联，
    不是靠相邻位置猜测。快照允许消费者自由处理，但消费者改动它不能回写 History。

    修改 `log_repetitions`，观察原始字节数如何增长。此处保护的是显式登记的约束文本，
    并不能自动识别任意自然语言中的所有约束。


In [3]:
log_repetitions = 80
history = ctx.History()
rule_id = history.append('instruction', '不得修改验证器。', protected=True)
history.append('user', '修复 status 筛选。')
call_id = history.append('call', 'run_tests()', call_id='test-1')
result_id = history.append('result', 'assert status failed\n' * log_repetitions, call_id='test-1')
history.append('user', '先检查字段名。')
snapshot = history.snapshot()
original_digest = ctx.digest(snapshot['items'])
snapshot['items'][0]['text'] = '错误地修改快照'
assert history.snapshot()['items'][0]['text'] == '不得修改验证器。'
assert ctx.digest(history.snapshot()['items']) == original_digest
{'revision': history.revision, 'raw_bytes': sum(len(i['text'].encode()) for i in history.snapshot()['items'])}


{'revision': 5, 'raw_bytes': 1759}

## 2. 预算按完整调用单元选择


    若只留工具输出而丢掉调用，就无法解释这份观察的来源。选择器把调用与结果作为一个单元；
    在这个小实现中，受保护指令和未闭合调用必须保留，其余单元从新到旧选取，再恢复时间顺序。
    它会整组省略长结果，不会把一份截断日志伪装成完整证据。

    调大 `budget_bytes` 使整对工具记录进入请求，再调小使它们一起离开。
    若必要内容本身超预算，应明确失败，不能悄悄丢掉规则。


In [4]:
budget_bytes = 160
projected = ctx.select(history.view(), budget_bytes)
ids = {i['id'] for i in projected['items']}
assert rule_id in ids
assert (call_id in ids) == (result_id in ids)
assert projected['used_bytes'] <= budget_bytes
assert ctx.digest(history.snapshot()['items']) == original_digest
failure = expect_error(ctx.Rejected, lambda: ctx.select(history.view(), 1), 'mandatory_context')
{'visible_roles': [i['role'] for i in projected['items']], 'omitted': projected['omitted'], 'too_small': failure}


{'visible_roles': ['instruction', 'user', 'user'],
 'omitted': ['e3', 'e4'],
 'too_small': 'mandatory_context_exceeds_budget'}

## 3. 摘要是带来源的候选，不能覆盖原文


    候选携带原始版本、来源摘要和覆盖范围，安装者核对这些结构后才发布。
    受保护原文独立保留；摘要的正文可以更短，但没有获得指令身份。
    修改 `summary_text` 可以改变压缩内容；结构检查不会判断其中每句话是否属实。
    这正是摘要质量评估仍然必要的原因。


In [5]:
summary_text = '一次测试发现 status 筛选失败，尚未修复。'
base = history.snapshot()
candidate = ctx.propose(base, summary_text)
history.install(candidate)
view = history.view()
assert view[0]['id'] == rule_id
assert view[-1]['sources'] == tuple(i['id'] for i in base['items'])
assert ctx.digest(history.snapshot()['items']) == original_digest
assert sum(len(i['text'].encode()) for i in view) < sum(len(i['text'].encode()) for i in base['items'])
{'model_view': view, 'raw_items_still_present': len(base['items'])}


{'model_view': [{'id': 'e1',
   'role': 'instruction',
   'text': '不得修改验证器。',
   'call_id': None,
   'protected': True},
  {'id': 'summary',
   'role': 'summary',
   'text': '一次测试发现 status 筛选失败，尚未修复。',
   'call_id': None,
   'protected': False,
   'sources': ('e1', 'e2', 'e3', 'e4', 'e5')}],
 'raw_items_still_present': 5}

## 4. 晚到、取消与重建：谁能安装结果


    摘要生成期间用户可能追加约束。旧候选即使内容合理，也不能覆盖新历史。
    这里采用严格版本相等：任何追加都拒绝旧候选，然后基于新快照重建。
    Codex 的真实安装还处理可接回的 suffix，规则比此实验更细。

    取消在安装入口检查；“不安装”不等于撤回已经发生的历史追加。


In [6]:
late = ctx.propose(history.snapshot(), '旧候选')
added_id = history.append('user', '保留现有排序规则。', protected=True)
stale = expect_error(ctx.Rejected, lambda: history.install(late), 'stale_revision')
fresh = ctx.propose(history.snapshot(), '测试失败；需修复字段且保留排序。')
cancelled = expect_error(ctx.Rejected, lambda: history.install(fresh, cancelled=True), 'cancelled')
assert history.projection is None
assert history.snapshot()['items'][-1]['id'] == added_id
history.install(fresh)
assert history.view()[1]['id'] == added_id
{'late_result': stale, 'cancelled_result': cancelled, 'installed_revision': fresh.base_revision}


{'late_result': 'stale_revision',
 'cancelled_result': 'cancelled',
 'installed_revision': 6}

## 5. 故意删规则或拆调用，验证拒绝依据


    两个反例分别触发不同不变量：缺少显式保护的原文、摘要覆盖范围拆开调用与结果。
    错误必须在安装前出现，不能先替换历史再试图补救。修改 `covered_count`，比较覆盖到调用和覆盖完整配对的差别。


In [7]:
from dataclasses import replace
good = ctx.propose(history.snapshot(), '可审查的摘要')
bad = replace(good, protected=())
missing = expect_error(ctx.Rejected, lambda: history.install(bad), 'protected_text')
covered_count = 3  # e1 指令、e2 用户、e3 调用；没有 e4 结果
split = ctx.propose(history.snapshot(), '不完整来源', count=covered_count)
pairing = expect_error(ctx.Rejected, lambda: history.install(split), 'split_tool_pair')
assert history.projection == fresh
{'missing_constraint': missing, 'broken_pair': pairing}


{'missing_constraint': 'missing_or_changed_protected_text',
 'broken_pair': 'summary_would_split_tool_pair'}

## 回到 Codex 源码

以下仅陈述固定提交 `c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee` 的静态源码事实；本 Notebook 没有执行 Rust 测试。Python 类型、字段、预算与简化策略属于教学设计，不能据此声称 Codex 使用同一实现。

- [ContextManager](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/context_manager/history.rs#L105-L146)：ContextManager 使用 Arc<Vec<ResponseItemEnvelope>> 形成写时复制快照；for_prompt 消费快照并规范化/替换不可用媒体。

- [ContextManager::record_annotated_items](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/context_manager/history.rs#L628-L642)：工具输出预算在 record_item_with_metadata 中施于 processed 副本；record_annotated_items 对原 envelope 保留来源元数据，session 随后持久化原 envelope。

- [build_and_install_compaction](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/compact_remote_v2_installation.rs#L83-L90)：V2 候选先核 model slug/comp_hash；最终 replacement 在 session state 锁内核 generation、prefix.len、auth_owner，再将 current[prefix.len..] 接回，随后调用 history replacement。


## 现在应能解释什么


    预算影响可见投影，不应破坏原始证据；摘要候选需要来源与安装版本；结构合法仍不保证语义正确。
    接回 Loop 时，History 继续拥有写入和安装权，请求构造器只读快照，摘要生成器只交候选。

    本章只验证内存算法和固定字节预算；没有验证真实模型摘要质量、精确 token 容量或持久恢复。
